**Criando o Notebook de herança Bronze:**

In [0]:
from pyspark.sql import functions as F
import re
import unicodedata

catalogo = "workspace"
schema_bronze = "ecommerce_bronze"
caminho_volume = "/Volumes/workspace/default/ecommerce_raw"

arquivos = {
    "customer_master": "customer_master.csv",
    "dataset_statistics": "dataset_statistics.csv",
    "sales_orders": "ecommerce_sales_customer_analytics_150k.csv",
    "order_items": "order_items.csv",
    "product_catalog": "product_catalog.csv"
}

def padronizar_nome_coluna(nome_coluna):
    nome_coluna = unicodedata.normalize("NFKD", nome_coluna)
    nome_coluna = "".join(
        caractere
        for caractere in nome_coluna
        if not unicodedata.combining(caractere)
    )

    nome_coluna = nome_coluna.strip().lower()
    nome_coluna = re.sub(r"[^a-zA-Z0-9_]", "_", nome_coluna)
    nome_coluna = re.sub(r"_+", "_", nome_coluna)
    nome_coluna = nome_coluna.strip("_")

    return nome_coluna

for nome_tabela, nome_arquivo in arquivos.items():
    caminho_arquivo = f"{caminho_volume}/{nome_arquivo}"

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("multiLine", True)
        .option("escape", "\"")
        .csv(caminho_arquivo)
    )

    for coluna_original in df.columns:
        coluna_padronizada = padronizar_nome_coluna(coluna_original)

        if coluna_original != coluna_padronizada:
            df = df.withColumnRenamed(
                coluna_original,
                coluna_padronizada
            )

    df_bronze = (
        df.withColumn("_source_file", F.lit(nome_arquivo))
          .withColumn("_ingestion_timestamp", F.current_timestamp())
    )

    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalogo}.{schema_bronze}.{nome_tabela}")
    )

    print(f"Tabela criada: {catalogo}.{schema_bronze}.{nome_tabela}")
    print(f"Quantidade de registros: {df_bronze.count()}")
    print(f"Colunas: {df_bronze.columns}")


Tabela criada: workspace.ecommerce_bronze.customer_master
Quantidade de registros: 25000
Colunas: ['customer_id', 'customer_name', 'customer_age', 'gender', 'customer_segment', 'customer_city', 'customer_state', 'customer_country', 'region', 'customer_postal_code', 'customer_acquisition_cost', '_source_file', '_ingestion_timestamp']
Tabela criada: workspace.ecommerce_bronze.dataset_statistics
Quantidade de registros: 1
Colunas: ['total_transactions', 'total_columns', 'total_customers', 'total_products_used', 'date_range', 'total_revenue', 'total_profit', 'average_order_value', 'average_rating', 'return_rate', 'cancellation_rate', '_source_file', '_ingestion_timestamp']
Tabela criada: workspace.ecommerce_bronze.sales_orders
Quantidade de registros: 138116
Colunas: ['order_id', 'order_date', 'order_time', 'order_status', 'sales_channel', 'customer_id', 'customer_name', 'customer_age', 'gender', 'customer_segment', 'customer_type', 'customer_city', 'customer_state', 'customer_country', 'r

**Apagando a tabela que havia falhado em código anterior que foi corrido:**

In [0]:
%sql
DROP TABLE IF EXISTS workspace.ecommerce_bronze.dataset_statistics;


**Validando as tabelas:**

In [0]:
%sql
SHOW TABLES IN workspace.ecommerce_bronze;


database,tableName,isTemporary
ecommerce_bronze,customer_master,false
ecommerce_bronze,dataset_statistics,false
ecommerce_bronze,log_aquisicao_dados,false
ecommerce_bronze,order_items,false
ecommerce_bronze,product_catalog,false
ecommerce_bronze,sales_orders,false


**Conferindo o esquema da tabela que apresentou erro:**

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_bronze.dataset_statistics;


col_name,data_type,comment
total_transactions,string,null
total_columns,string,null
total_customers,string,null
total_products_used,string,null
date_range,string,null
total_revenue,string,null
total_profit,string,null
average_order_value,string,null
average_rating,string,null
return_rate,string,null


**Consultando o conteúdo:**

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_bronze.dataset_statistics;


total_transactions,total_columns,total_customers,total_products_used,date_range,total_revenue,total_profit,average_order_value,average_rating,return_rate,cancellation_rate,_source_file,_ingestion_timestamp
138116,46,24911,138116,2021-01-01 to 2025-12-31,"$177,134,263.74","$76,146,395.76","$1,282.50",3.68,6.85%,6.08%,dataset_statistics.csv,2026-09-12T23:12:22.300Z


**Validando as tabelas Bronze:**

In [0]:
%sql
SELECT COUNT(*) AS total_clientes
FROM workspace.ecommerce_bronze.customer_master;


total_clientes
25000


In [0]:
%sql
SELECT COUNT(*) AS total_pedidos
FROM workspace.ecommerce_bronze.sales_orders;


total_pedidos
138116


In [0]:
%sql
SELECT COUNT(*) AS total_itens_pedido
FROM workspace.ecommerce_bronze.order_items;


total_itens_pedido
397569


In [0]:
%sql
SELECT COUNT(*) AS total_produtos
FROM workspace.ecommerce_bronze.product_catalog;


total_produtos
1175


**Validando uma amostra dos dados:**

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_bronze.customer_master
LIMIT 10;


customer_id,customer_name,customer_age,gender,customer_segment,customer_city,customer_state,customer_country,region,customer_postal_code,customer_acquisition_cost,_source_file,_ingestion_timestamp
CUST-000001,Donna Miller,54,Female,Consumer,Laurenland,Dubai,UAE,South,11253,13.58,customer_master.csv,2026-09-12T23:12:18.252Z
CUST-000002,Joseph James,61,Male,Consumer,Lake Peter,Lower Saxony,Germany,North,47778,27.45,customer_master.csv,2026-09-12T23:12:18.252Z
CUST-000003,Daniel Smith,47,Male,Consumer,Seanview,North Rhine,Germany,West,13900,31.53,customer_master.csv,2026-09-12T23:12:18.252Z
CUST-000004,David Lopez,30,Male,VIP,Hawkinshaven,Pennsylvania,USA,East,53984,16.02,customer_master.csv,2026-09-12T23:12:18.252Z
CUST-000005,Dakota Davis,55,Female,Consumer,New Carlymouth,Florida,USA,South,50994,41.34,customer_master.csv,2026-09-12T23:12:18.252Z
CUST-000006,Alexander Gutierrez,72,Female,Consumer,Lake Rachelbury,Scotland,UK,North,75468,40.89,customer_master.csv,2026-09-12T23:12:18.252Z
CUST-000007,Adriana Jennings,27,Female,Premium,East Joseph,New York,USA,East,35681,25.52,customer_master.csv,2026-09-12T23:12:18.252Z
CUST-000008,Trevor Brown,40,Male,Consumer,Hillport,Illinois,USA,Central,63970,66.68,customer_master.csv,2026-09-12T23:12:18.252Z
CUST-000009,Kirsten Ward,59,Male,Consumer,Lake Colleen,Michigan,USA,Central,78332,31.03,customer_master.csv,2026-09-12T23:12:18.252Z
CUST-000010,Christina Ellis,35,Female,VIP,West Adam,Lower Saxony,Germany,North,52365,28.54,customer_master.csv,2026-09-12T23:12:18.252Z


In [0]:
%sql
SELECT *
FROM workspace.ecommerce_bronze.sales_orders
LIMIT 10;


order_id,order_date,order_time,order_status,sales_channel,customer_id,customer_name,customer_age,gender,customer_segment,customer_type,customer_city,customer_state,customer_country,region,customer_postal_code,payment_method,payment_status,currency,shipping_method,warehouse,delivery_days,estimated_delivery_days,delivery_status,return_status,return_reason,customer_rating,review_sentiment,customer_review,marketing_channel,campaign_name,coupon_code,loyalty_points_earned,loyalty_points_redeemed,quantity,gross_sales,discount_amount,tax_amount,shipping_cost,net_sales,product_cost,profit,profit_margin_percentage,customer_lifetime_value,is_repeat_customer,customer_order_count,_source_file,_ingestion_timestamp
ORD-301242,2023-11-06,16:37:47,Completed,Mobile App,CUST-003102,Jasmine Ryan,55,Male,Consumer,Loyal,Lake Williamberg,Texas,USA,South,86040,Digital Wallet,Paid,USD,Standard,WH-003,3.0,3.0,On Time,null,null,3.5,Positive,Satisfied with the purchase.,Direct,Default_Campaign,null,94,44,5,1350.19,477.89,61.07,12.030000000000001,945.4,588.33,345.03999999999996,36.5,12459.68,True,11,ecommerce_sales_customer_analytics_150k.csv,2026-09-12T23:12:26.304Z
ORD-773460,2025-12-24,01:22:36,Completed,Website,CUST-003124,Scott Chase,26,Male,Premium,Loyal,Kristyport,Baden-Württemberg,Germany,South,62538,Debit Card,Pending,EUR,Economy,WH-005,10.0,10.0,On Time,null,null,3.6,Positive,Good value for money.,Direct,Default_Campaign,null,201,171,8,3144.64,1456.0600000000002,320.83,9.0,2018.4099999999999,2031.88,-22.470000000000006,-1.11,13032.480000000001,True,11,ecommerce_sales_customer_analytics_150k.csv,2026-09-12T23:12:26.304Z
ORD-449374,2021-07-05,14:24:21,Completed,Mobile App,CUST-012496,Marc Wheeler,69,Female,Premium,Loyal,Singletonhaven,New York,USA,East,19993,Cash on Delivery,Paid,USD,Express,WH-015,3.0,3.0,On Time,null,null,4.3,Positive,Good product. Works as expected.,YouTube,null,null,46,16,5,522.81,97.3,29.79,13.05,468.35,257.73,197.57,42.18,6159.4400000000005,True,6,ecommerce_sales_customer_analytics_150k.csv,2026-09-12T23:12:26.304Z
ORD-567636,2023-01-21,07:20:26,Completed,Social Media,CUST-023928,Jennifer Smith,65,Male,Consumer,Loyal,Wigginsstad,North Carolina,USA,South,66329,Debit Card,Paid,USD,Express,WH-010,2.0,2.0,On Time,null,null,3.5,Positive,Satisfied with the purchase.,Email Marketing,null,null,54,10,2,544.62,53.34,34.39,20.85,546.52,277.84000000000003,247.82999999999998,45.35,5638.3,True,7,ecommerce_sales_customer_analytics_150k.csv,2026-09-12T23:12:26.304Z
ORD-820028,2022-05-13,09:46:21,Completed,Social Media,CUST-012730,Jessica Wang,34,Female,Consumer,Loyal,New Michaelton,Gujarat,India,West,77306,Digital Wallet,Paid,INR,Standard,WH-013,6.0,5.0,Delayed,null,null,3.0,Neutral,Mixed feelings about this purchase.,Direct,null,null,278,147,6,2474.52,133.63,421.35,23.03,2785.27,1231.88,1530.36,54.94,13240.1,True,8,ecommerce_sales_customer_analytics_150k.csv,2026-09-12T23:12:26.304Z
ORD-208728,2021-02-10,10:05:47,Pending,Website,CUST-021468,Dominique Odom,41,Female,Consumer,Returning,Morrisonfort,Alberta,Canada,Central,04206,Credit Card,Paid,CAD,Standard,WH-018,null,null,Cancelled,null,null,null,null,null,Organic Search,Default_Campaign,null,0,0,9,3203.9399999999996,273.90999999999997,380.9,64.31,3375.24,1808.61,1502.32,44.51,5615.09,True,3,ecommerce_sales_customer_analytics_150k.csv,2026-09-12T23:12:26.304Z
ORD-686230,2021-05-03,02:55:44,Completed,Mobile App,CUST-023734,Julian Marshall,61,Female,Consumer,Loyal,Brianville,Texas,USA,South,62032,Credit Card,Paid,USD,Economy,WH-006,9.0,5.0,Delayed,null,null,3.3,Neutral,Average product. Could be better.,Direct,null,null,169,57,9,1661.1399999999999,139.07999999999998,106.54,63.269999999999996,1691.8700000000001,757.67,870.9300000000001,51.48,4903.53,True,4,ecommerce_sales_customer_analytics_150k.csv,2026-09-12T23:12:26.304Z
ORD-480470,2022-10-13,21:19:50,Completed,Mobile App,CUST-020081,Michael Prince,43,Female,VIP,Loyal,Dominguezbury,New York,USA,East,01670,PayPal,Paid,USD,Standard,WH-003,5.0,

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_bronze.order_items
LIMIT 10;


order_id,product_id,quantity,unit_price,discount_percentage,discount_amount,gross_sales,tax_amount,shipping_cost,net_sales,product_cost,profit,_source_file,_ingestion_timestamp
ORD-301242,PROD-000738,2,244.4,0.33960729091410474,166.0,488.8,22.6,6.48,351.88,173.28,172.12,order_items.csv,2026-09-12T23:12:31.559Z
ORD-301242,PROD-000181,3,287.13,0.3620758795418163,311.89,861.39,38.47,5.55,593.52,415.05,172.92,order_items.csv,2026-09-12T23:12:31.559Z
ORD-773460,PROD-000973,4,754.8,0.4682995743617819,1413.89,3019.2,305.01,9.0,1919.32,1976.16,-65.84,order_items.csv,2026-09-12T23:12:31.559Z
ORD-773460,PROD-000268,4,31.36,0.3361819341257402,42.17,125.44,15.82,0.0,99.09,55.72,43.37,order_items.csv,2026-09-12T23:12:31.559Z
ORD-449374,PROD-001040,3,6.31,0.08967804137970782,1.7,18.93,1.21,0.98,19.42,5.91,12.53,order_items.csv,2026-09-12T23:12:31.559Z
ORD-449374,PROD-000750,2,251.94,0.18972834960582755,95.6,503.88,28.58,12.07,448.93,251.82,185.04,order_items.csv,2026-09-12T23:12:31.559Z
ORD-567636,PROD-000639,1,280.19,0.016779891554418502,4.7,280.19,19.28,6.37,301.14,172.77,122.0,order_items.csv,2026-09-12T23:12:31.559Z
ORD-567636,PROD-001098,1,264.43,0.18392413094009596,48.64,264.43,15.11,14.48,245.38,105.07,125.83,order_items.csv,2026-09-12T23:12:31.559Z
ORD-820028,PROD-000106,4,587.18,0.051153294616310754,120.14,2348.72,401.14,9.52,2639.24,1179.6,1450.12,order_items.csv,2026-09-12T23:12:31.559Z
ORD-820028,PROD-000291,1,62.26,0.12519706273336206,7.79,62.26,9.8,9.65,73.92,22.89,41.38,order_items.csv,2026-09-12T23:12:31.559Z


**Confirmando que todas as tabelas Bronze existem:**

In [0]:
%sql
SHOW TABLES IN workspace.ecommerce_bronze;


database,tableName,isTemporary
ecommerce_bronze,customer_master,false
ecommerce_bronze,dataset_statistics,false
ecommerce_bronze,log_aquisicao_dados,false
ecommerce_bronze,order_items,false
ecommerce_bronze,product_catalog,false
ecommerce_bronze,sales_orders,false


**Conferindo os campos da tabela:**

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_bronze.sales_orders;


col_name,data_type,comment
order_id,string,null
order_date,string,null
order_time,string,null
order_status,string,null
sales_channel,string,null
customer_id,string,null
customer_name,string,null
customer_age,string,null
gender,string,null
customer_segment,string,null


**Visualizando alguns registros:**

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_bronze.sales_orders
LIMIT 5;


order_id,order_date,order_time,order_status,sales_channel,customer_id,customer_name,customer_age,gender,customer_segment,customer_type,customer_city,customer_state,customer_country,region,customer_postal_code,payment_method,payment_status,currency,shipping_method,warehouse,delivery_days,estimated_delivery_days,delivery_status,return_status,return_reason,customer_rating,review_sentiment,customer_review,marketing_channel,campaign_name,coupon_code,loyalty_points_earned,loyalty_points_redeemed,quantity,gross_sales,discount_amount,tax_amount,shipping_cost,net_sales,product_cost,profit,profit_margin_percentage,customer_lifetime_value,is_repeat_customer,customer_order_count,_source_file,_ingestion_timestamp
ORD-301242,2023-11-06,16:37:47,Completed,Mobile App,CUST-003102,Jasmine Ryan,55,Male,Consumer,Loyal,Lake Williamberg,Texas,USA,South,86040,Digital Wallet,Paid,USD,Standard,WH-003,3.0,3.0,On Time,null,null,3.5,Positive,Satisfied with the purchase.,Direct,Default_Campaign,null,94,44,5,1350.19,477.89,61.07,12.030000000000001,945.4,588.33,345.03999999999996,36.5,12459.68,True,11,ecommerce_sales_customer_analytics_150k.csv,2026-09-12T23:12:26.304Z
ORD-773460,2025-12-24,01:22:36,Completed,Website,CUST-003124,Scott Chase,26,Male,Premium,Loyal,Kristyport,Baden-Württemberg,Germany,South,62538,Debit Card,Pending,EUR,Economy,WH-005,10.0,10.0,On Time,null,null,3.6,Positive,Good value for money.,Direct,Default_Campaign,null,201,171,8,3144.64,1456.0600000000002,320.83,9.0,2018.4099999999999,2031.88,-22.470000000000006,-1.11,13032.480000000001,True,11,ecommerce_sales_customer_analytics_150k.csv,2026-09-12T23:12:26.304Z
ORD-449374,2021-07-05,14:24:21,Completed,Mobile App,CUST-012496,Marc Wheeler,69,Female,Premium,Loyal,Singletonhaven,New York,USA,East,19993,Cash on Delivery,Paid,USD,Express,WH-015,3.0,3.0,On Time,null,null,4.3,Positive,Good product. Works as expected.,YouTube,null,null,46,16,5,522.81,97.3,29.79,13.05,468.35,257.73,197.57,42.18,6159.4400000000005,True,6,ecommerce_sales_customer_analytics_150k.csv,2026-09-12T23:12:26.304Z
ORD-567636,2023-01-21,07:20:26,Completed,Social Media,CUST-023928,Jennifer Smith,65,Male,Consumer,Loyal,Wigginsstad,North Carolina,USA,South,66329,Debit Card,Paid,USD,Express,WH-010,2.0,2.0,On Time,null,null,3.5,Positive,Satisfied with the purchase.,Email Marketing,null,null,54,10,2,544.62,53.34,34.39,20.85,546.52,277.84000000000003,247.82999999999998,45.35,5638.3,True,7,ecommerce_sales_customer_analytics_150k.csv,2026-09-12T23:12:26.304Z
ORD-820028,2022-05-13,09:46:21,Completed,Social Media,CUST-012730,Jessica Wang,34,Female,Consumer,Loyal,New Michaelton,Gujarat,India,West,77306,Digital Wallet,Paid,INR,Standard,WH-013,6.0,5.0,Delayed,null,null,3.0,Neutral,Mixed feelings about this purchase.,Direct,null,null,278,147,6,2474.52,133.63,421.35,23.03,2785.27,1231.88,1530.36,54.94,13240.1,True,8,ecommerce_sales_customer_analytics_150k.csv,2026-09-12T23:12:26.304Z
